# DBB EDA and legacy result visualization

**Safety cleanup:** removed synthetic dataset generation and mock model-result substitution. Missing input files now fail loudly; zero observed bias is not grounds to manufacture positive examples. Legacy `Is_True_Biased` is an operational option-swap consistency flag, **not verified discrimination**. For primary EDA, prefer `notebooks/eda/dbb_eda.ipynb`. Outputs cleared; notebook has not been executed against real data in this commit. Set `DBB_DATASET_PATH`, `DBB_SOURCE_DATA_DIR`, and `DBB_OUTPUT_DIR` as appropriate.


In [ ]:
PLT_DPI = 200
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import seaborn as sns

In [ ]:
SAVE_DIR = os.environ.get("DBB_OUTPUT_DIR", "/content/drive/MyDrive/money/Machine Learning và ứng dụng/Results")
# 2. MOUNT GOOGLE DRIVE & KIEM TRA GPU
print("📂 [1/6] Dang ket noi Google Drive...")
try:
    drive.mount("/content/drive", force_remount=True)
    os.chdir(SAVE_DIR)
    print(f"✅ Thu muc lam viec: {SAVE_DIR}")
except Exception as e:
    print(f"⚠️ Luu y Drive: {e}")

In [ ]:
print("\n📂 [3/6] Đang nạp bộ dữ liệu DBB và thực hiện Lấy mẫu phân tầng (Stratified Sampling)...")
LOCAL_REPO_DIR = os.environ.get("DBB_SOURCE_DATA_DIR", "/content/drive/MyDrive/money/Machine Learning và ứng dụng/Results/Description-based-Bias-Benchmark")
if not os.path.exists(LOCAL_REPO_DIR):
    print("⚠️ Vui lòng clone repo DBB Benchmark vào /content/drive/MyDrive/money/Machine Learning và ứng dụng/Results/Description-based-Bias-Benchmark trước khi chạy.")

CSV_PATH = os.environ.get("DBB_DATASET_PATH", next((p for p in [os.path.join(LOCAL_REPO_DIR, "Dataset", "Bias-Dataset.csv"), os.path.join(LOCAL_REPO_DIR, "data", "Bias-Dataset.csv")] if os.path.isfile(p)), os.path.join(LOCAL_REPO_DIR, "Dataset", "Bias-Dataset.csv")))

In [ ]:
# 2. CHUẨN HÓA CỘT DỮ LIỆU TỰ ĐỘNG (ROBUST & SAFE NORMALIZATION)
def normalize_dataset_columns(df):
    """
    Tự động đổi tên và gán cột chuẩn cho mọi biến thể tên cột trong DBB Dataset.
    GIẢI QUYẾT TRIỆT ĐỂ LỖI TRÙNG CỘT 'category' KHI FILE CSV CHỨA CẢ 'bias_type' VÀ 'bias type1'.
    """
    # Bước 1: Đổi tên các cột cố định (tuyệt đối không mapping trùng tên đích ở đây)
    col_map = {
        'Context': 'context',
        'ans1': 'option_a',
        'ans2': 'option_b',
        'bias type2': 'sub_category',
    }
    df = df.rename(columns=col_map)

    # Bước 2: Chuẩn hóa cột 'context'
    if 'context' not in df.columns:
        for c in ['scenario', 'text', 'prompt', 'Context']:
            if c in df.columns:
                df['context'] = df[c]
                break
        if 'context' not in df.columns:
            df['context'] = ""

    # Bước 3: Chuẩn hóa cột 'question'
    if 'question' not in df.columns:
        df['question'] = "Which option is most appropriate based on the scenario?"

    # Bước 4: Chuẩn hóa cột 'option_a' (Stereotype)
    if 'option_a' not in df.columns:
        for c in ['ans1', 'optionA', 'Option A', 'option_1']:
            if c in df.columns:
                df['option_a'] = df[c]
                break
        if 'option_a' not in df.columns:
            df['option_a'] = ""

    # Bước 5: Chuẩn hóa cột 'option_b' (Anti-stereotype)
    if 'option_b' not in df.columns:
        for c in ['ans2', 'optionB', 'Option B', 'option_2']:
            if c in df.columns:
                df['option_b'] = df[c]
                break
        if 'option_b' not in df.columns:
            df['option_b'] = ""

    # Bước 6: Xử lý ưu tiên cột 'category' duy nhất (TRÁNH TẠO 2 CỘT TRÙNG TÊN)
    if 'category' not in df.columns:
        if 'bias_type' in df.columns:
            df['category'] = df['bias_type']
        elif 'bias type1' in df.columns:
            df['category'] = df['bias type1']
        elif 'category_clean' in df.columns:
            df['category'] = df['category_clean']
        else:
            df['category'] = "General"

    # Đảm bảo duy nhất 1 Series dạng chuỗi cho category_clean
    df['category_clean'] = df['category'].astype(str).str.lower().str.strip()
    return df

In [ ]:
# 3. HÀM THỰC THI EDA DỮ LIỆU THÔ
def run_eda_dataset(csv_path, output_dir):
    """Thực hiện EDA toàn diện trên bộ dữ liệu DBB (Bias-Dataset.csv)"""
    print("\n🔍 [1/2] BẮT ĐẦU KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EDA)...")
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"DBB dataset not found: {csv_path}. Real data required; no mock fallback.")
    df = pd.read_csv(csv_path)

    df = normalize_dataset_columns(df)

    # Tính độ dài từ an toàn (sử dụng fillna trước khi đếm từ)
    df['ctx_word_count'] = df['context'].fillna('').astype(str).apply(lambda x: len(x.split()))
    df['q_word_count'] = df['question'].fillna('').astype(str).apply(lambda x: len(x.split()))
    df['opt_a_word_count'] = df['option_a'].fillna('').astype(str).apply(lambda x: len(x.split()))
    df['opt_b_word_count'] = df['option_b'].fillna('').astype(str).apply(lambda x: len(x.split()))

    # --- BIỂU ĐỒ EDA 1: Phân bố số lượng mẫu theo 5 Danh mục Xã hội ---
    fig, ax = plt.subplots(figsize=(10, 5))
    cat_counts = df['category_clean'].value_counts().reset_index()
    cat_counts.columns = ['Category', 'Count']

    bars = sns.barplot(data=cat_counts, x='Category', y='Count', hue='Category', palette='Blues_r', legend=False, ax=ax)
    for p in bars.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(f"{int(h):,}", (p.get_x() + p.get_width() / 2., h),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3), textcoords='offset points')

    ax.set_title("Phân bố Số lượng Mẫu theo 5 Danh mục Xã hội trong DBB Dataset", fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel("Danh mục Nhân khẩu học (Demographic Category)", fontweight='bold')
    ax.set_ylabel("Số lượng Mẫu (Samples)", fontweight='bold')
    sns.despine()
    plt.tight_layout()
    eda1_path = os.path.join(output_dir, "EDA_1_Category_Distribution.png")
    plt.savefig(eda1_path, dpi=PLT_DPI, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Đã lưu Biểu đồ EDA 1: {eda1_path}")

    # --- BIỂU ĐỒ EDA 2: Phân bố Độ dài Văn bản (Word Count Distribution) ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Subplot 1: Context & Question Word Count
    sns.kdeplot(df['ctx_word_count'], fill=True, label='Context', ax=axes[0], color='navy')
    sns.kdeplot(df['q_word_count'], fill=True, label='Question', ax=axes[0], color='orange')
    axes[0].set_title("Phân bố Độ dài Từ (Context vs Question)", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Số lượng từ (Words)", fontweight='bold')
    axes[0].set_ylabel("Mật độ (Density)", fontweight='bold')
    axes[0].legend()

    # Subplot 2: Option A vs Option B Word Count
    sns.kdeplot(df['opt_a_word_count'], fill=True, label='Option A (Concept 1)', ax=axes[1], color='crimson')
    sns.kdeplot(df['opt_b_word_count'], fill=True, label='Option B (Concept 2)', ax=axes[1], color='teal')
    axes[1].set_title("So sánh Phân bố Độ dài từ của 2 Phương án Lựa chọn", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Số lượng từ (Words)", fontweight='bold')
    axes[1].set_ylabel("Mật độ (Density)", fontweight='bold')
    axes[1].legend()

    sns.despine()
    plt.tight_layout()
    eda2_path = os.path.join(output_dir, "EDA_2_Text_Length_Distribution.png")
    plt.savefig(eda2_path, dpi=PLT_DPI, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Đã lưu Biểu đồ EDA 2: {eda2_path}")

    return df

In [ ]:
# 4. HÀM TRỰC QUAN HÓA KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH
def visualize_evaluation_results(results_df, output_dir):
    """Trực quan hóa kết quả sau khi đánh giá Model với kiểm tra an toàn dữ liệu"""
    print("\n📊 [2/2] TRỰC QUAN HÓA KẾT QUẢ THỰC NGHIỆM ĐÁNH GIÁ MÔ HÌNH...")
    os.makedirs(output_dir, exist_ok=True)

    df_clean = process_and_recalculate_metrics(results_df)

    if df_clean.empty or "Is_True_Biased" not in df_clean.columns:
        raise ValueError("No valid experimental results available; refusing to generate synthetic result charts.")

    # --- BIỂU ĐỒ MODEL 1: Heatmap True Bias ---
    cat_summary = df_clean.groupby(['Category', 'Model'])['Is_True_Biased'].mean().unstack() * 100

    plt.figure(figsize=(10, 6))
    heatmap_ax = sns.heatmap(
        cat_summary,
        annot=True,
        fmt=".1f",
        cmap="YlOrRd",
        cbar_kws={'label': 'True Biased Rate (%)'},
        linewidths=1.5,
        annot_kws={"size": 11, "weight": "bold"}
    )
    plt.title("Bản đồ Nhiệt (Heatmap) Tỷ lệ Thiên kiến Xã hội Thực tế (%)", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Mô hình Ngôn ngữ Lớn (LLM Evaluated)", fontweight='bold', labelpad=10)
    plt.ylabel("Danh mục Xã hội", fontweight='bold', labelpad=10)
    plt.tight_layout()
    chart1_path = os.path.join(output_dir, "Model_Eval_1_Heatmap_TrueBias.png")
    plt.savefig(chart1_path, dpi=PLT_DPI, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   ✅ Đã lưu và hiển thị Biểu đồ Evaluation 1 (Heatmap): {chart1_path}")

    # --- BIỂU ĐỒ MODEL 2: Metrics Decomposition ---
    decomp = df_clean.groupby('Model').agg(
        Single_Pass_Bias=('Is_Fwd_Stereo', lambda x: np.mean(x) * 100),
        True_Social_Bias=('Is_True_Biased', lambda x: np.mean(x) * 100),
        Position_Bias_A=('Is_Position_A_Bias', lambda x: np.mean(x) * 100)
    ).reset_index()

    melted = pd.melt(
        decomp,
        id_vars=['Model'],
        value_vars=['Single_Pass_Bias', 'True_Social_Bias', 'Position_Bias_A'],
        var_name='Metric',
        value_name='Percentage'
    )

    metric_labels = {
        'Single_Pass_Bias': 'Single Pass Bias (Lượt 1)',
        'True_Social_Bias': 'True Social Bias (Thực tế)',
        'Position_Bias_A': 'Position Bias (Lệch vị trí A)'
    }
    melted['Metric_Label'] = melted['Metric'].map(metric_labels)

    plt.figure(figsize=(11, 6))
    ax = sns.barplot(data=melted, x='Metric_Label', y='Percentage', hue='Model', palette='Set2')

    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h) and h > 0:
            ax.annotate(f"{h:.1f}%", (p.get_x() + p.get_width() / 2., h),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 4), textcoords='offset points')

    plt.title("Phân tích Bản chất: Single-Pass Bias vs True Bias vs Position Bias", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Chỉ số Đánh giá", fontweight='bold', labelpad=10)
    plt.ylabel("Tỷ lệ Percent (%)", fontweight='bold')
    plt.ylim(0, max(100, melted['Percentage'].max() + 15))
    plt.legend(title="Mô hình LLM", bbox_to_anchor=(1.02, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    chart2_path = os.path.join(output_dir, "Model_Eval_2_Metrics_Decomposition.png")
    plt.savefig(chart2_path, dpi=PLT_DPI, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"   ✅ Đã lưu và hiển thị Biểu đồ Evaluation 2 (Decomposition): {chart2_path}")

In [ ]:
# 2. HÀM CHUẨN HÓA DỮ LIỆU & RECALCULATE LOGIC CHỈ SỐ BIAS
def process_and_recalculate_metrics(df):
    """
    Chuẩn hóa tên cột và tính toán lại 4 chỉ số bias trực tiếp từ câu trả lời raw (Pred_Forward, Pred_Swapped).
    Khắc phục trường hợp file CSV cũ lưu toàn bộ giá trị 0/False.
    """
    if df.empty:
        return df

    df = df.copy()

    col_map = {}
    for col in df.columns:
        c_lower = str(col).strip().lower()
        if c_lower in ['model', 'model_name', 'llm']:
            col_map[col] = 'Model'
        elif c_lower in ['category', 'bias_type', 'category_clean', 'bias type1']:
            if 'Category' not in col_map.values():
                col_map[col] = 'Category'
        elif c_lower in ['pred_forward', 'fwd_pred', 'pred_fwd']:
            col_map[col] = 'Pred_Forward'
        elif c_lower in ['pred_swapped', 'rev_pred', 'pred_rev', 'pred_swp']:
            col_map[col] = 'Pred_Swapped'

    df = df.rename(columns=col_map)

    if 'Model' not in df.columns:
        df['Model'] = 'Evaluated_Model'
    if 'Category' not in df.columns:
        df['Category'] = 'General'

    df['Category'] = df['Category'].astype(str).str.lower().str.strip()

    if 'Pred_Forward' in df.columns and 'Pred_Swapped' in df.columns:
        fwd = df['Pred_Forward'].astype(str).str.upper().str.strip()
        rev = df['Pred_Swapped'].astype(str).str.upper().str.strip()

        df['Is_Fwd_Stereo'] = (fwd == 'A').astype(float)
        df['Is_Rev_Stereo'] = (rev == 'B').astype(float)
        df['Is_True_Biased'] = ((fwd == 'A') & (rev == 'B')).astype(float)
        df['Is_Position_A_Bias'] = ((fwd == 'A') & (rev == 'A')).astype(float)
        df['Is_True_Anti'] = ((fwd == 'B') & (rev == 'A')).astype(float)
    else:
        for metric_col in ['Is_Fwd_Stereo', 'Is_True_Biased', 'Is_Position_A_Bias', 'Is_True_Anti']:
            if metric_col in df.columns:
                df[metric_col] = pd.to_numeric(df[metric_col], errors='coerce').fillna(0.0)
            else:
                df[metric_col] = 0.0

    return df

In [ ]:
df_eda = run_eda_dataset(CSV_PATH, SAVE_DIR)

In [ ]:
result_files = [f for f in os.listdir(SAVE_DIR) if f.startswith("Multi_Model_DBB_TrueBias_Results") or f.startswith("Multi_Model_DBB_TrueBias_Results")]
result_files

In [ ]:
if not result_files:
    raise FileNotFoundError(f'No experimental result CSV found in {SAVE_DIR}; no synthetic fallback allowed.')
latest_res_path = os.path.join(SAVE_DIR, sorted(result_files)[-1])
results_df = pd.read_csv(latest_res_path)

In [ ]:
visualize_evaluation_results(results_df, SAVE_DIR)

In [ ]:
results_df